# z602 - Feature Engineering (Etapa 2)
Campo clave: `product_id`. Fuente: `sell-in-zeroes.txt.gz` (ya generado por z601, no se recalculan ceros).
Corre en Jupyter sobre Google Cloud (sin mount de Drive).

In [12]:
import os
import numpy as np
import polars as pl
import warnings
warnings.filterwarnings("ignore")

In [13]:
PARAM = {
    'experimento': 'FE601',
    'base_path': './buckets/b1/datasets/',
    'archivo_ceros': 'sell-in-zeroes.txt',
    'windows': [3, 6, 9, 12, 18, 24, 36],
    'lags': [1, 2, 3, 6, 12],
    'umbral_joven_meses': 12
}

In [14]:
# creo la carpeta del experimento
ruta = os.path.join('./exp', PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

./exp/FE601


In [11]:
!find . -iname "sell-in-zeroes*"

./buckets/b1/datasets/sell-in-zeroes.txt


## 1. Carga y agregado a product_id

In [16]:
# el archivo sell-in-zeroes.txt.gz fue generado con DuckDB COPY (FORMAT csv),
# separador coma y header por defecto -- NO es tab como el sell-in original
dataset = pl.read_csv(
    os.path.join(PARAM['base_path'], PARAM['archivo_ceros']),
    separator=","
)
print(dataset.columns)
print(dataset.height)

['customer_id', 'product_id', 'periodo', 'plan_precios_cuidados', 'cust_request_qty', 'cust_request_tn', 'tn']
11155579


In [17]:
# agrego a nivel product_id x periodo (campo clave = product_id)
# el grid customer x product x periodo ya viene completo dentro de la vida del producto,
# por lo tanto sumar sobre customer_id ya deja una grilla completa por producto, sin gaps
tb_prod = dataset.group_by(["product_id", "periodo"]).agg(
    pl.col("tn").sum().alias("tn")
).sort(["product_id", "periodo"])

print(tb_prod.height)

31522


## 2. Lags

In [18]:
df = tb_prod.clone()

# lag: si el producto no existia en ese periodo, shift() deja null automaticamente
# porque no hay fila previa dentro de ese grupo (over product_id)
lag_exprs = [
    pl.col("tn").shift(n).over("product_id").alias(f"tn_lag_{n}")
    for n in PARAM['lags']
]
df = df.with_columns(lag_exprs)

## 3. Rolling (media, varianza, max, min, suma)
Todas las ventanas se calculan sobre `tn` desplazado 1 periodo, para que el periodo actual NUNCA entre en su propia ventana (evita fuga de informacion).

In [19]:
df = df.with_columns(
    pl.col("tn").shift(1).over("product_id").alias("tn_shift1")
)

rolling_exprs = []
for w in PARAM['windows']:
    base = pl.col("tn_shift1")
    rolling_exprs += [
        base.rolling_mean(window_size=w, min_periods=1).over("product_id").alias(f"tn_media_{w}"),
        base.rolling_std(window_size=w, min_periods=2).over("product_id").pow(2).alias(f"tn_var_{w}"),
        base.rolling_max(window_size=w, min_periods=1).over("product_id").alias(f"tn_max_{w}"),
        base.rolling_min(window_size=w, min_periods=1).over("product_id").alias(f"tn_min_{w}"),
        base.rolling_sum(window_size=w, min_periods=1).over("product_id").alias(f"tn_suma_{w}"),
    ]
df = df.with_columns(rolling_exprs)

In [20]:
# indice de sobrecompra/stockeo: suma de ultimos 3 meses vs promedio anual (x3 para escala comparable)
df = df.with_columns(
    (pl.col("tn_suma_3") / (pl.col("tn_media_12") * 3 + 1e-6)).alias("ratio_sobrecompra_3_12")
)

## 4. Patron de frecuencia de compra
Calculado con un loop secuencial por producto (780 series, tamano trivial). Cada valor en la fila `i` usa SOLO datos hasta `i-1`.

In [21]:
def periodo_a_meses(periodo: int) -> int:
    return (periodo // 100) * 12 + (periodo % 100)

df = df.with_columns(
    pl.col("periodo").map_elements(periodo_a_meses, return_dtype=pl.Int64).alias("periodo_m")
)

nacimiento = df.group_by("product_id").agg(pl.col("periodo_m").min().alias("nacimiento_m"))
df = df.join(nacimiento, on="product_id")

df = df.with_columns(
    (pl.col("periodo_m") - pl.col("nacimiento_m")).alias("edad_producto")
)
df = df.with_columns(
    (pl.col("edad_producto") < PARAM['umbral_joven_meses']).cast(pl.Int8).alias("producto_joven")
)

In [22]:
def features_frecuencia(g: pl.DataFrame) -> pl.DataFrame:
    tn = g["tn"].to_numpy()
    periodo_m = g["periodo_m"].to_numpy()
    n = len(tn)

    meses_desde_ultima_compra = np.full(n, np.nan)
    racha_actual_sin_compra = np.full(n, np.nan)
    mayor_racha_historica = np.full(n, np.nan)
    promedio_gap_historico = np.full(n, np.nan)
    tn_promedio_por_compra_hist = np.full(n, np.nan)

    ultima_compra_idx = None
    racha_actual = 0
    max_racha = 0
    gaps = []
    tn_compras = []

    for i in range(n):
        # --- lectura: solo estado acumulado HASTA i-1 ---
        if ultima_compra_idx is not None:
            meses_desde_ultima_compra[i] = periodo_m[i] - periodo_m[ultima_compra_idx]
        racha_actual_sin_compra[i] = racha_actual
        mayor_racha_historica[i] = max_racha
        if gaps:
            promedio_gap_historico[i] = float(np.mean(gaps))
        if tn_compras:
            tn_promedio_por_compra_hist[i] = float(np.mean(tn_compras))

        # --- actualizacion: incorporo el periodo actual, para la fila siguiente ---
        if tn[i] > 0:
            if ultima_compra_idx is not None:
                gap = periodo_m[i] - periodo_m[ultima_compra_idx] - 1
                gaps.append(gap)
            tn_compras.append(tn[i])
            ultima_compra_idx = i
            racha_actual = 0
        else:
            racha_actual += 1
            max_racha = max(max_racha, racha_actual)

    return g.with_columns([
        pl.Series("meses_desde_ultima_compra", meses_desde_ultima_compra),
        pl.Series("racha_actual_sin_compra", racha_actual_sin_compra),
        pl.Series("mayor_racha_historica", mayor_racha_historica),
        pl.Series("promedio_gap_historico", promedio_gap_historico),
        pl.Series("tn_promedio_por_compra_hist", tn_promedio_por_compra_hist),
    ])

df = df.sort(["product_id", "periodo"])
df = df.group_by("product_id", maintain_order=True).map_groups(features_frecuencia)

## 5. Test de fuga de outliers
Trucho un valor absurdo (5.000.000.000) en un periodo intermedio de UN producto y verifico que las filas ANTERIORES a ese periodo no cambian (no debe haber fuga hacia atras en el tiempo).

In [23]:
producto_test = df["product_id"][0]
sub = df.filter(pl.col("product_id") == producto_test).sort("periodo")
periodo_medio = sub["periodo"][sub.height // 2]

tb_prod_trucho = tb_prod.clone()
tb_prod_trucho = tb_prod_trucho.with_columns(
    pl.when((pl.col("product_id") == producto_test) & (pl.col("periodo") == periodo_medio))
    .then(5_000_000_000.0)
    .otherwise(pl.col("tn"))
    .alias("tn")
)

# recalculo solo la media movil 12 sobre este producto, mismo criterio (shift(1) antes del rolling)
chk = tb_prod_trucho.filter(pl.col("product_id") == producto_test).sort("periodo")
chk = chk.with_columns(pl.col("tn").shift(1).alias("tn_shift1"))
chk = chk.with_columns(
    pl.col("tn_shift1").rolling_mean(window_size=12, min_periods=1).alias("tn_media_12_trucho")
)

original = sub.select(["periodo", "tn_media_12"])
comparacion = chk.select(["periodo", "tn_media_12_trucho"]).join(original, on="periodo", how="left")

antes = comparacion.filter(pl.col("periodo") < periodo_medio)
diff = (antes["tn_media_12_trucho"] - antes["tn_media_12"]).abs().max()
print("periodo truchado:", periodo_medio)
print("maxima diferencia en periodos ANTERIORES (debe ser 0 o null):", diff)
assert diff == 0 or diff is None, "FUGA DETECTADA: un valor futuro esta afectando el pasado"

periodo truchado: 201807
maxima diferencia en periodos ANTERIORES (debe ser 0 o null): 0.0


## 6. Guardar dataset

In [24]:
salida = os.path.join(ruta, "tb_features_FE601.parquet")
df.write_parquet(salida)
print(salida)
print(df.shape)

./exp/FE601/tb_features_FE601.parquet
(31522, 54)
